# 📊 Telco Customer Churn — Keşifsel Veri Analizi (EDA)

**Amaç:** Bu rapor, Telekom sektöründeki müşteri kaybının (churn) arkasındaki dinamikleri görselleştirmek, istatistiksel olarak anlamlı örüntüleri keşfetmek ve model geliştirme öncesinde domain bilgisi oluşturmak için hazırlanmıştır.

---

In [ ]:
# Kütüphaneler
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Stil ayarları
sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

COLORS = ['#3498db', '#e74c3c']
PALETTE = {'No': '#3498db', 'Yes': '#e74c3c'}

df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].replace(' ', np.nan), errors='coerce').fillna(0)
print(f'Veri Seti Boyutu: {df.shape[0]:,} satır × {df.shape[1]} sütun')

## 1️⃣ Veri Setine Genel Bakış

In [ ]:
print('İlk 5 Satır:')
display(df.head())
print('\nVeri Tipleri ve Eksik Değerler:')
info_df = pd.DataFrame({
    'Veri Tipi': df.dtypes,
    'Eksik Değer': df.isnull().sum(),
    'Eksik (%)': (df.isnull().sum() / len(df) * 100).round(2),
    'Unique': df.nunique()
})
display(info_df)
print('\nSayısal İstatistikler:')
display(df.describe().round(2))

## 2️⃣ Hedef Değişken Analizi (Target Distribution)
Müşteri kaybı oranımız nedir? Veri dengeli mi?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pasta Grafiği
churn_counts = df['Churn'].value_counts()
axes[0].pie(churn_counts, labels=['Kalan (No)', 'Kayıp (Yes)'],
            autopct='%1.1f%%', colors=COLORS, startangle=90,
            explode=(0, 0.08), shadow=True,
            textprops={'fontsize': 13, 'fontweight': 'bold'})
axes[0].set_title('Müşteri Kaybı Dağılımı', fontsize=15, fontweight='bold')

# Bar Grafiği
sns.countplot(x='Churn', data=df, palette=PALETTE, ax=axes[1], edgecolor='black')
for p in axes[1].patches:
    axes[1].annotate(f'{int(p.get_height()):,}', (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontsize=13, fontweight='bold')
axes[1].set_title('Churn Sayısal Dağılım', fontsize=15, fontweight='bold')
axes[1].set_xlabel('')

plt.tight_layout()
plt.savefig('static/01_churn_distribution.png', bbox_inches='tight')
plt.show()
print(f'\n⚠️ Dengesizlik Oranı: {churn_counts["Yes"]/churn_counts["No"]:.2f}x — '
      f'SMOTE uygulanması gerekir.')

## 3️⃣ Sözleşme Tipi vs Churn (En Kritik Değişken)
Aylık sözleşmeli müşteriler ile uzun dönemli müşteriler arasındaki farkı inceleyelim.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sayısal
sns.countplot(x='Contract', hue='Churn', data=df, palette=PALETTE, ax=axes[0], edgecolor='black')
axes[0].set_title('Sözleşme Tipine Göre Churn', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Sözleşme Tipi')
axes[0].legend(title='Churn')

# Yüzdesel
ct = pd.crosstab(df['Contract'], df['Churn'], normalize='index') * 100
ct.plot(kind='bar', stacked=True, color=COLORS, ax=axes[1], edgecolor='black')
axes[1].set_title('Sözleşme Tipine Göre Churn Oranı (%)', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Oran (%)')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
axes[1].legend(title='Churn')

plt.tight_layout()
plt.savefig('static/02_contract_vs_churn.png', bbox_inches='tight')
plt.show()
print('\n💡 INSIGHT: Aylık sözleşmeli müşterilerin ~%42 si ayrılıyor. '
      '2 yıllık sözleşmelerde bu oran sadece ~%3.')

## 4️⃣ Bağlılık Süresi (Tenure) Analizi
Müşteriler ne zaman ayrılıyor?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# KDE Plot
for label, color in zip(['No', 'Yes'], COLORS):
    sns.kdeplot(df[df['Churn'] == label]['tenure'], fill=True, color=color,
                label=f'{"Kalan" if label == "No" else "Kayıp"} ({label})', ax=axes[0])
axes[0].set_title('Tenure Yoğunluk Dağılımı', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Abonelik Süresi (Ay)')
axes[0].legend()

# Boxplot
sns.boxplot(x='Churn', y='tenure', data=df, palette=PALETTE, ax=axes[1])
axes[1].set_title('Tenure Kutu Grafiği (Outlier Analizi)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('static/03_tenure_analysis.png', bbox_inches='tight')
plt.show()
print('\n💡 INSIGHT: Kayıp müşterilerin büyük çoğunluğu ilk 1-10 ay içinde ayrılıyor. '
      'İlk yıl kritik bir elde tutma dönemidir.')

## 5️⃣ Aylık ve Toplam Ücret Analizi

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# MonthlyCharges
for label, color in zip(['No', 'Yes'], COLORS):
    sns.kdeplot(df[df['Churn'] == label]['MonthlyCharges'], fill=True, color=color,
                label=f'{"Kalan" if label == "No" else "Kayıp"} ({label})', ax=axes[0])
axes[0].set_title('Aylık Ücret Dağılımı', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Aylık Ücret ($)')
axes[0].legend()

# TotalCharges
for label, color in zip(['No', 'Yes'], COLORS):
    sns.kdeplot(df[df['Churn'] == label]['TotalCharges'], fill=True, color=color,
                label=f'{"Kalan" if label == "No" else "Kayıp"} ({label})', ax=axes[1])
axes[1].set_title('Toplam Ücret Dağılımı', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Toplam Ücret ($)')
axes[1].legend()

plt.tight_layout()
plt.savefig('static/04_charges_analysis.png', bbox_inches='tight')
plt.show()
print('\n💡 INSIGHT: Yüksek aylık ücret ödeyen müşteriler daha çok gidiyor. '
      'Ancak toplam harcaması düşük olanlar (yeni müşteriler) en riskli grup.')

## 6️⃣ Korelasyon Analizi (Heatmap)
Sayısal değişkenler arasındaki ilişkiyi inceleyelim.

In [ ]:
# Sayısal sütunları al
df_numeric = df[['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']].copy()
df_numeric['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

fig, ax = plt.subplots(figsize=(8, 6))
corr = df_numeric.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, cmap='RdBu_r', center=0,
            fmt='.2f', linewidths=1, ax=ax, square=True,
            cbar_kws={'shrink': 0.8})
ax.set_title('Korelasyon Matrisi (Heatmap)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('static/05_correlation_heatmap.png', bbox_inches='tight')
plt.show()
print('\n💡 INSIGHT: tenure ile TotalCharges arasında güçlü pozitif korelasyon var (beklenen). '
      'MonthlyCharges ile Churn arasında pozitif korelasyon dikkat çekici.')

## 7️⃣ Kategorik Değişkenler ve Churn İlişkisi
Hangi hizmetler müşteri kaybını etkiliyor?

In [ ]:
cat_features = ['InternetService', 'PaymentMethod', 'OnlineSecurity',
                'TechSupport', 'PaperlessBilling', 'Partner']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, col in enumerate(cat_features):
    ct = pd.crosstab(df[col], df['Churn'], normalize='index') * 100
    ct.plot(kind='bar', stacked=True, color=COLORS, ax=axes[idx], edgecolor='black', legend=False)
    axes[idx].set_title(f'{col} vs Churn (%)', fontweight='bold')
    axes[idx].set_ylabel('Churn Oranı (%)')
    axes[idx].set_xticklabels(axes[idx].get_xticklabels(), rotation=45, ha='right')

# Ortak Legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, ['Kalan (No)', 'Kayıp (Yes)'], loc='upper right', fontsize=12)
plt.suptitle('Kategorik Değişkenlerin Churn Oranına Etkisi', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('static/06_categorical_churn.png', bbox_inches='tight')
plt.show()
print('\n💡 INSIGHT: Fiber optik internet, elektronik çek ve destek hizmeti olmayanlar en riskli gruplar.')

## 8️⃣ İstatistiksel Anlamlılık Testi (Chi-Square)
Kategorik değişkenlerin Churn ile ilişkisi istatistiksel olarak anlamlı mı?

In [ ]:
# Chi-square test — Hipotez testi
cat_cols = df.select_dtypes(include='object').columns.drop('customerID')
chi2_results = []

for col in cat_cols:
    if col == 'Churn':
        continue
    contingency = pd.crosstab(df[col], df['Churn'])
    chi2, p_value, dof, expected = stats.chi2_contingency(contingency)
    chi2_results.append({
        'Değişken': col,
        'Chi² Değeri': round(chi2, 2),
        'P-Değeri': f'{p_value:.2e}',
        'Anlamlı mı? (p<0.05)': '✅ Evet' if p_value < 0.05 else '❌ Hayır'
    })

chi2_df = pd.DataFrame(chi2_results).sort_values('Chi² Değeri', ascending=False)
print('Chi-Square Bağımsızlık Testi Sonuçları:')
print('=' * 70)
display(chi2_df)
print('\n💡 INSIGHT: Contract, InternetService ve OnlineSecurity en yüksek Chi² değerine sahip — '
      'Churn ile en güçlü ilişkiyi bu değişkenler kurur.')

## 9️⃣ Sonuç ve Öneriler

### Temel Bulgular:
| # | Bulgu | Aksiyon |
| :-- | :-- | :-- |
| 1 | **Aylık sözleşmeli** müşterilerin %42'si ayrılıyor | Uzun dönem teşvikleri sunulmalı |
| 2 | **İlk 12 ay** en kritik dönem | Onboarding programı güçlendirilmeli |
| 3 | **Fiber optik** kullanıcıları daha çok gidiyor | Hizmet kalitesi incelenmeli |
| 4 | **Destek hizmeti olmayanlar** daha riskli | Cross-sell fırsatı |
| 5 | **Elektronik çek** ödeme yöntemi riskli | Otomatik ödemeye geçiş teşviki |
| 6 | Veri **dengesiz** (%26 Churn) | SMOTE ile dengeleme uygulandı |

---
*Bu analiz `02_model_training.py` dosyasındaki model seçimi için temel oluşturmuştur.*